In [101]:
import pandas as pd
import plotly.express as px

In [102]:
abilities_df = pd.read_excel("../data/Abilities.xlsx")

skills_df = pd.read_excel("../data/Essential Skills.xlsx")

In [103]:
abilities_selected = [
    "Oral Expression",
    "Deductive Reasoning"
]

abilities_filtered = abilities_df[
    abilities_df["Element Name"].isin(abilities_selected)
]

In [104]:
skills_selected = [
    "Critical Thinking",
    "Active Learning"
]

skills_filtered = skills_df[
    skills_df["Element Name"].isin(skills_selected)
]

In [105]:
combined_df = pd.concat([
    abilities_filtered,
    skills_filtered
])

In [106]:
combined_df["Element Name"].unique()

<ArrowStringArray>
[    'Oral Expression', 'Deductive Reasoning',   'Critical Thinking',
     'Active Learning']
Length: 4, dtype: str

In [107]:
pivot_df = combined_df.pivot_table(
    index="Title",
    columns="Element Name",
    values="Data Value",
    aggfunc="mean"
).reset_index()

In [108]:
pivot_df.columns

Index(['Title', 'Active Learning', 'Critical Thinking', 'Deductive Reasoning',
       'Oral Expression'],
      dtype='str', name='Element Name')

In [109]:
pivot_df["AI Enhancement Score"] = (
    pivot_df["Critical Thinking"] * 0.35 +
    pivot_df["Active Learning"] * 0.30 +
    pivot_df["Deductive Reasoning"] * 0.20 +
    pivot_df["Oral Expression"] * 0.15
)

In [110]:
top_ai = pivot_df.sort_values(
    by="AI Enhancement Score",
    ascending=False
)

top_ai.head(15)

Element Name,Title,Active Learning,Critical Thinking,Deductive Reasoning,Oral Expression,AI Enhancement Score
460,"Judges, Magistrate Judges, and Magistrates",4.310,5.315,4.810,4.625,4.80900
651,Physicists,4.750,4.500,4.810,4.935,4.70225
76,Biochemists and Biophysicists,4.685,4.685,4.500,4.810,4.66675
676,Preventive Medicine Physicians,4.440,4.685,4.750,4.880,4.65375
514,Mathematicians,4.560,4.560,4.500,4.620,4.55700
425,Hospitalists,4.500,4.435,4.560,4.685,4.51700
267,Emergency Medicine Physicians,4.190,4.565,4.630,4.870,4.51125
469,"Law Teachers, Postsecondary",4.375,4.440,4.440,5.000,4.50450
588,Obstetricians and Gynecologists,4.185,4.690,4.565,4.625,4.50375
243,"Education Administrators, Kindergarten through...",4.060,4.560,4.690,4.940,4.49300


In [123]:
fig = px.scatter(

    pivot_df,

    x="Critical Thinking",
    y="Active Learning",

    color="Cluster",

    size="AI Enhancement Score",

    hover_name="Title",

    hover_data={
        "Critical Thinking": True,
        "Active Learning": True,
        "AI Enhancement Score": ":.2f",
        "Cluster": True
    },

    template="plotly_dark",

    title="AI Workforce Clusters"
)

In [126]:
fig.update_traces(
    marker=dict(
        line=dict(width=1, color="white")
    ),
    opacity=0.85
)

In [127]:
fig.update_layout(

    title={
        "text": "AI Workforce Clusters",
        "x": 0.5,
        "xanchor": "center"
    },

    title_font_size=28,

    xaxis_title="Critical Thinking",

    yaxis_title="Active Learning",

    plot_bgcolor="#050816",

    paper_bgcolor="#050816",

    font=dict(
        family="Arial",
        color="white",
        size=14
    ),

    legend_title="Cluster",

    height=800
)

In [128]:
fig.show()

In [113]:
from sklearn.cluster import KMeans


In [114]:
features = pivot_df[[
    "Critical Thinking",
    "Active Learning",
    "Deductive Reasoning",
    "Oral Expression"
]]

In [115]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

In [116]:
pivot_df["Cluster"] = kmeans.fit_predict(features)

In [117]:
pivot_df.head()

Element Name,Title,Active Learning,Critical Thinking,Deductive Reasoning,Oral Expression,AI Enhancement Score,Cluster
0,Accountants and Auditors,3.310,3.935,4.125,4.065,3.80500,0
1,Actors,2.620,3.000,2.940,4.060,3.03300,3
2,Actuaries,3.625,4.500,4.440,4.375,4.20675,2
3,Acupuncturists,3.060,3.750,3.880,3.940,3.59750,0
4,Acute Care Nurses,4.125,4.060,4.120,4.250,4.12000,2


In [129]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [130]:
features = pivot_df[[
    "Critical Thinking",
    "Active Learning",
    "Deductive Reasoning",
    "Oral Expression"
]]

In [131]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

In [132]:
pca = PCA(n_components=2)

In [133]:
pca_result = pca.fit_transform(scaled_features)

In [134]:
pivot_df["PCA1"] = pca_result[:, 0]
pivot_df["PCA2"] = pca_result[:, 1]

In [135]:
fig = px.scatter(

    pivot_df,

    x="PCA1",
    y="PCA2",

    color="Cluster",

    size="AI Enhancement Score",

    hover_name="Title",

    template="plotly_dark",

    title="Human + AI Cognitive Landscape"
)

In [136]:
fig.update_traces(
    marker=dict(
        line=dict(width=1, color="white")
    ),
    opacity=0.85
)

In [137]:
fig.update_layout(

    title={
        "text": "Human + AI Cognitive Landscape",
        "x": 0.5
    },

    title_font_size=30,

    xaxis_title="Cognitive Dimension 1",

    yaxis_title="Cognitive Dimension 2",

    plot_bgcolor="#050816",

    paper_bgcolor="#050816",

    font=dict(
        family="Arial",
        color="white",
        size=14
    ),

    height=850
)

In [138]:
fig.show()

In [139]:
pivot_df.groupby("Cluster")["Title"].apply(list).head()

Cluster
0    [Accountants and Auditors, Acupuncturists, Adm...
1    [Adhesive Bonding Machine Operators and Tender...
2    [Actuaries, Acute Care Nurses, Adapted Physica...
3    [Actors, Agricultural Technicians, Aircraft Ca...
Name: Title, dtype: object

In [140]:
for cluster in sorted(pivot_df["Cluster"].unique()):

    print(f"\nCLUSTER {cluster}\n")

    display(

        pivot_df[
            pivot_df["Cluster"] == cluster
        ][[
            "Title",
            "AI Enhancement Score"
        ]]

        .sort_values(
            by="AI Enhancement Score",
            ascending=False
        )

        .head(10)
    )


CLUSTER 0



Element Name,Title,AI Enhancement Score
444,Information Technology Project Managers,3.94175
215,Database Administrators,3.91525
159,Computer Network Architects,3.91375
743,Sales Managers,3.91150
225,Detectives and Criminal Investigators,3.91100
868,Validation Engineers,3.91050
887,Wind Energy Engineers,3.90800
277,Environmental Engineering Technologists and Te...,3.90250
230,"Directors, Religious Activities and Education",3.90200
196,Credit Analysts,3.89900



CLUSTER 1



Element Name,Title,AI Enhancement Score
516,Mechanical Door Repairers,2.99300
176,Continuous Mining Machine Operators,2.97325
471,"Layout Workers, Metal and Plastic",2.97250
195,Crane and Tower Operators,2.96050
377,Gem and Diamond Workers,2.95050
760,Septic Tank Servicers and Sewer Pipe Cleaners,2.94750
194,Craft Artists,2.94750
677,Print Binding and Finishing Workers,2.94575
758,Semiconductor Processing Technicians,2.94325
766,Sheet Metal Workers,2.94000



CLUSTER 2



Element Name,Title,AI Enhancement Score
460,"Judges, Magistrate Judges, and Magistrates",4.80900
651,Physicists,4.70225
76,Biochemists and Biophysicists,4.66675
676,Preventive Medicine Physicians,4.65375
514,Mathematicians,4.55700
425,Hospitalists,4.51700
267,Emergency Medicine Physicians,4.51125
469,"Law Teachers, Postsecondary",4.50450
588,Obstetricians and Gynecologists,4.50375
243,"Education Administrators, Kindergarten through...",4.49300



CLUSTER 3



Element Name,Title,AI Enhancement Score
326,"First-Line Supervisors of Landscaping, Lawn Se...",3.44550
394,Graphic Designers,3.43925
596,Online Merchants,3.43525
496,Magnetic Resonance Imaging Technologists,3.43300
716,Recycling Coordinators,3.43150
56,Audio and Video Technicians,3.42675
504,Manufactured Building and Mobile Home Installers,3.41775
858,Travel Agents,3.41625
22,Aircraft Cargo Handling Supervisors,3.41575
94,Broadcast Technicians,3.41450


In [150]:
fig.write_html("../visuals/ai_workforce_clusters.html")

In [152]:
pivot_df.to_csv(
    "../data/ai_workforce_analysis.csv",
    index=False
)